# Test

#### Comparing two .csv files on "WSI_Name" and display the unique names

In [ ]:
import pandas as pd

# 1. Read both files
df1 = pd.read_csv('/data_64T_3/Shared/Extracted_Features/TRIDENT/TCGA_KIRC/Tumor_DX1/feature_extraction_metrics_resnet18.csv')
df2 = pd.read_csv('/data_64T_3/Shared/Extracted_Features/TRIDENT/TCGA_KIRC/Tumor_DX1/tile_extraction_metrics.csv')

# 2. Convert columns to sets for comparison
slides_1 = set(df1['slide_name'])
slides_2 = set(df2['slide_name'])

# 3. Find differences
in_1_not_2 = list(slides_1 - slides_2)
in_2_not_1 = list(slides_2 - slides_1)

# 4. Format and Print results
print(f"--- Slides in Feature Metrics (df1) but MISSING in Tile Metrics (df2) [Count: {len(in_1_not_2)}] ---")
if in_1_not_2:
    print(", ".join([f"'{name}'" for name in in_1_not_2]))
else:
    print("None")

print(f"\n--- Slides in Tile Metrics (df2) but MISSING in Feature Metrics (df1) [Count: {len(in_2_not_1)}] ---")
if in_2_not_1:
    print(", ".join([f"'{name}'" for name in in_2_not_1]))
else:
    print("None")

#### Finding unique "WSI_Name" from multiple .csv files 

In [5]:
import pandas as pd
from functools import reduce

tissue = "Normal_TS"   # Normal_TS, Tumor_DX1
dataset = "TCGA_STAD"

# file1 = f"/data_64T_3/Shared/Extracted_Features/TRIDENT/{dataset}/{tissue}/feature_extraction_metrics_resnet50_1024.csv"
# file2 = f"/data_64T_3/Shared/Extracted_Features/HISTOLAB/{dataset}/{tissue}/feature_extraction_metrics_resnet50_1024.csv"
# file3 = f"/data_64T_3/Shared/Extracted_Features/CLAM/{dataset}/{tissue}/feature_extraction_metrics_resnet50_1024.csv"
# file4 = "/data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Feature_extraction_metrics_uni.csv"

file1 = "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_CLAM.csv"
file2 = "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_HISTOLAB.csv"
file3 = "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_MUFASA.csv"
file4 = "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_TRIDENT.csv"

# Load CSV files
files = [file1, file2, file3, file4]
wsi_sets = [set(pd.read_csv(f)['slide_name'].dropna()) for f in files]

# Find common WSI_Names across all files
common = reduce(lambda a, b: a & b, wsi_sets)

# Find uncommon WSI_Names for each file
for i, (f, wsi_set) in enumerate(zip(files, wsi_sets), 1):
    uncommon = wsi_set - common
    if uncommon:
        print(f"\n{f} - Unique to this file ({len(uncommon)}):")
        print(sorted(uncommon))

# Summary statistics
print(f"\n--- Summary ---")
print(f"Common across all files: {len(common)}")
for i, (f, wsi_set) in enumerate(zip(files, wsi_sets), 1):
    print(f"{f}: {len(wsi_set)} total, {len(wsi_set - common)} unique")


/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_CLAM.csv - Unique to this file (11):
['TCGA-44-8119-01Z-00-DX1.1EBEBFA7-22DB-4365-9DF8-C4E679C11312', 'TCGA-44-A479-01Z-00-DX1.CA5654C6-A623-452E-A8AC-DB6279CA97B1', 'TCGA-44-A47B-01Z-00-DX1.177D0531-E037-435B-BFD4-382B2150B10D', 'TCGA-44-A47F-01Z-00-DX1.C1EC3F2D-33C8-483B-9DDA-2F01B5FED618', 'TCGA-44-A47G-01Z-00-DX1.810F67EC-3A0C-4056-B736-3331A11412CC', 'TCGA-49-4487-01Z-00-DX1.3a3a0720-463c-430e-849b-e2f8991bdfa5', 'TCGA-49-6767-01Z-00-DX1.53459c0e-b8ec-4893-9910-87b63c503134', 'TCGA-55-8505-01Z-00-DX1.D364C30D-BFB8-486B-A0D3-948FF8E90C3E', 'TCGA-55-A48X-01Z-00-DX1.A46C6373-8458-4D55-88C3-4C70A05F9F47', 'TCGA-62-A470-01Z-00-DX1.3983A979-E756-413C-941A-C8B83AD3BBA7', 'TCGA-95-8039-01Z-00-DX1.38b07435-987d-495d-9940-ba5f20e6ef97']

/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUAD_HISTOLAB.csv - Unique to this file (12):
['TCGA-44-8119-01Z-00-DX1.1EBEBFA7-22DB-4365-9DF8-C4E679C11312', 'TCGA-44-A479-01Z-00-DX1.CA5654C6-A623-452E-A8AC-DB6279CA97B1', 'TCGA-44-A

##### storing common slide names in a csv file

In [7]:
import pandas as pd
from functools import reduce
from pathlib import Path

# --- Configuration ---
dataset = "TCGA_STAD"
tissue = "Normal_TS"   # Normal_TS, Tumor_DX1

# Define your input files
files = [
    "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUSC_CLAM.csv",
    "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUSC_HISTOLAB.csv", 
    "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUSC_TRIDENT.csv"
]

# Output configuration
output_csv = "/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUSC_common_slides.csv"
SAVE_COMMON = True  # <--- SWITCH: Set to True to save, False to skip

# --- Processing ---
# Load CSV files and extract sets of slide_names
print("Loading files...")
wsi_sets = []
for f in files:
    try:
        df = pd.read_csv(f)
        # Ensure we are using strings and drop NaNs
        names = set(df['slide_name'].dropna().astype(str))
        wsi_sets.append(names)
    except FileNotFoundError:
        print(f"Error: File not found: {f}")
        wsi_sets.append(set())

# Find common WSI_Names across all files
if wsi_sets:
    common = reduce(lambda a, b: a & b, wsi_sets)
else:
    common = set()

# Print discrepancies (Uncommon WSI_Names)
for i, (f, wsi_set) in enumerate(zip(files, wsi_sets), 1):
    uncommon = wsi_set - common
    if uncommon:
        print(f"\n[File {i}] {Path(f).name} - Unique to this file ({len(uncommon)}):")
        # print(sorted(list(uncommon))[:5]) # Uncomment to see first 5 names
        print(f"   (Hidden {len(uncommon)} names for brevity)")
    else:
        print(f"\n[File {i}] {Path(f).name} - No unique slides (all are in common set).")

# Summary statistics
print(f"\n--- Summary ---")
print(f"Total Common across all {len(files)} files: {len(common)}")
for i, (f, wsi_set) in enumerate(zip(files, wsi_sets), 1):
    print(f"File {i}: {len(wsi_set)} total, {len(wsi_set - common)} unique/missing from intersection")

# --- Switch Facility to Save ---
if SAVE_COMMON:
    if common:
        # Sort for consistency
        common_list = sorted(list(common))
        df_common = pd.DataFrame({'slide_name': common_list})
        df_common.to_csv(output_csv, index=False)
        print(f"\n[✓] Successfully saved {len(common)} common slides to '{output_csv}'")
    else:
        print("\n[!] No common slides found. Nothing to save.")
else:
    print(f"\n[i] Save switch is OFF. Common slides were NOT saved.")

Loading files...

[File 1] TCGA_LUSC_CLAM.csv - Unique to this file (4):
   (Hidden 4 names for brevity)

[File 2] TCGA_LUSC_HISTOLAB.csv - Unique to this file (4):
   (Hidden 4 names for brevity)

[File 3] TCGA_LUSC_TRIDENT.csv - No unique slides (all are in common set).

--- Summary ---
Total Common across all 3 files: 473
File 1: 477 total, 4 unique/missing from intersection
File 2: 477 total, 4 unique/missing from intersection
File 3: 473 total, 0 unique/missing from intersection

[✓] Successfully saved 473 common slides to '/data_64T_3/Raja/TCGA_NSCLC/TCGA_LUSC_common_slides.csv'


#### Getting uninque values given a list of values

In [76]:
string_list = ['SHS-11-42923@A1-1',
'SHS-10-27868@A3-1',
'SHS-08-30422@B3-1',
'SHS-08-43828@C4-1',
'SHS-10-34184@A4-1',
'SHS-12-26663@B11-1',
'SHS-08-30422@B4-1',
'SHS-10-03459@A2-1',
'SHS-10-11243@C-1',
'SHS-10-07754@A3-1',
'SHS-11-20040@C5-1', 
'SHS-10-07754@A4-1', 
'SHS-08-30422@B20-1']
unique_list = list(set(string_list))
unique_list

['SHS-08-30422@B4-1',
 'SHS-10-11243@C-1',
 'SHS-10-07754@A4-1',
 'SHS-08-30422@B20-1',
 'SHS-10-07754@A3-1',
 'SHS-11-42923@A1-1',
 'SHS-08-43828@C4-1',
 'SHS-08-30422@B3-1',
 'SHS-10-27868@A3-1',
 'SHS-10-03459@A2-1',
 'SHS-11-20040@C5-1',
 'SHS-12-26663@B11-1',
 'SHS-10-34184@A4-1']

#### Find missing values in .csv based on given input folder

In [13]:
import pandas as pd
from pathlib import Path

# Configuration
dir_path = '/data_64T_3/Shared/Extracted_Features/TRIDENT/TCGA_BRCA/Normal_TS/Extracted_features/resnet18'
csv_file = '/data_64T_3/Shared/Extracted_Features/TRIDENT/TCGA_BRCA/Normal_TS/tile_extraction_metrics.csv'

# Get file names from directory (without extension)
file_names = {f.stem for f in Path(dir_path).iterdir() if f.is_file()}

# Get WSI_Name from CSV
wsi_names = set(pd.read_csv(csv_file)['slide_name'].dropna())

# Find missing
missing = file_names - wsi_names

print(f"Files in directory: {len(file_names)}")
print(f"WSI_Names in CSV: {len(wsi_names)}")
print(f"Missing from CSV: {len(missing)}")
if missing:
    print("\nMissing file names:")
    print(sorted(missing))

Files in directory: 339
WSI_Names in CSV: 339
Missing from CSV: 0


#### Finding missing folders given a .csv file based on "WSI_Name"

In [1]:
import pandas as pd
from pathlib import Path

# Configuration
csv_path = "/data_64T_3/Raja/Extracted_Features/MUFASA/TCGA_COAD/Tumor_DX1/tile_extraction_metrics.csv"
folder_path = "/data_64T_3/Raja/Extracted_Features/MUFASA/TCGA_COAD/Tumor_DX1/Extracted_Features/uni"
csv_column = "slide_name"

# Load and compare
df = pd.read_csv(csv_path)
csv_names = set(df[csv_column].astype(str))
folder_names = set(p.name for p in Path(folder_path).iterdir() if p.is_dir())

# Find matches and mismatches
matched = csv_names & folder_names
csv_only = csv_names - folder_names
folders_only = folder_names - csv_names

# Report
print(f"✓ Matched: {len(matched)}/{len(csv_names)}")
print(f"✗ In CSV but not folders: {len(csv_only)}")
print(f"✗ In folders but not CSV: {len(folders_only)}")

if csv_only:
    print(f"\nMissing folders (first 10): {list(csv_only)[:10]}")
if folders_only:
    print(f"\nExtra folders (first 10): {list(folders_only)[:10]}")

✓ Matched: 255/261
✗ In CSV but not folders: 6
✗ In folders but not CSV: 0

Missing folders (first 10): ['TCGA-AA-3986-01Z-00-DX1.db60e495-c0eb-416c-b65b-55ce62ed10b0', 'TCGA-AA-3950-01Z-00-DX1.2a81cf11-4c16-4e9e-8809-6f63152060da', 'TCGA-AZ-4616-01Z-00-DX1.0a0f6eaa-4db6-4479-a9df-f09387f555b1', 'TCGA-D5-6531-01Z-00-DX1.32241731-5890-424e-96d5-b897e770f03c', 'TCGA-CM-6674-01Z-00-DX1.4a08b16a-788e-43dc-85d2-baff6e911de2', 'TCGA-F4-6703-01Z-00-DX1.28225f5d-d880-4605-831f-f22ec0272cde']


In [12]:
from pathlib import Path

path1 = Path("/data_64T_2/Dataset/TCGA_BLCA/images/Tumor_DX1")
path2 = Path("/data_64T_3/Raja/Extracted_Features/MUFASA/TCGA_BLCA/Tumor_DX1/Extracted_Features/uni")

# Get set of folder names from path2
folders = {p.name for p in path2.iterdir() if p.is_dir()}

# Find files in path1 whose stem (name without extension) is not in the folder set
missing = [f.name for f in path1.iterdir() if f.is_file() and f.stem not in folders]

print(missing)

['TCGA-5N-A9KI-01Z-00-DX1.1BAE5EFF-859F-4D0A-8DDC-527E2901F7D4.svs', 'TCGA-BT-A3PJ-01Z-00-DX1.C730AD35-81DF-4758-97B5-1E9F58E67A2B.svs', 'TCGA-DK-A6B0-01Z-00-DX1.83628045-FDD9-4167-879F-7813FDCA7D97.svs', 'TCGA-GC-A6I3-01Z-00-DX1.1B3D09B9-14FB-4236-9354-3E5CC59354A7.svs', 'TCGA-E7-A6MF-01Z-00-DX1.2B0C8CBF-3244-4D17-B9CA-3CBE000BEC29.svs', 'TCGA-E7-A3Y1-01Z-00-DX1.D3E81A0B-1D20-4916-8872-81D469A1E276.svs', 'TCGA-FD-A6TI-01Z-00-DX1.A451F82D-983B-4646-BFC6-E40A66F2B9C8.svs', 'TCGA-PQ-A6FN-01Z-00-DX1.58FC60E8-7386-4E57-804D-6AD103AE0FDC.svs', 'TCGA-4Z-AA84-01Z-00-DX1.604B60F3-1E19-4335-9F8D-0D6920B1ACF8.svs', 'TCGA-K4-A83P-01Z-00-DX1.6B9251BD-650E-4F69-BE8F-AB0BE5DC90FB.svs', 'TCGA-CF-A5UA-01Z-00-DX1.7352D4EB-46F5-4EAA-95B9-1D869E8291C3.svs', 'TCGA-E7-A4XJ-01Z-00-DX1.70DA8872-34A6-41B7-8D17-2B15C80EF5CF.svs', 'TCGA-FT-A3EE-01Z-00-DX1.1CA5BFA3-F6F7-45DE-83EC-F328AC4EBE80.svs', 'TCGA-CF-A8HY-01Z-00-DX1.53F599FE-3A6D-4BAC-92B2-CBE3F183BB71.svs', 'TCGA-FD-A3SP-01Z-00-DX1.05986238-A4B9-497D-9D0

#### Removing entries based on "WSI_Name" in .csv file

In [11]:
import pandas as pd
log_tile_extraction_path = "/data_64T_3/Shared/Extracted_Features/TRIDENT/TCGA_BRCA/Normal_TS/feature_extraction_metrics_uni.csv"
df = pd.read_csv(log_tile_extraction_path)
names_to_remove = ['TCGA-AC-A2FG-11A-01-TS1.8192AEB9-1A2C-42B1-BC91-FDBDB10FDF6F', 'TCGA-E9-A1RC-11A-01-TSA.eb86cac2-b5e3-479b-aecf-e5ee73bc5d77', 'TCGA-E9-A1NG-11A-03-TSC.275eda98-a0ca-4678-b254-f225074ec5b5', 'TCGA-E9-A1R7-11A-03-TSC.6c0e6f40-8a68-40b1-a29d-79b304fff3c9', 'TCGA-BH-A5IZ-11A-01-TS1.9345C1EC-2A27-49F1-AE61-7B056A982A34', 'TCGA-E2-A158-11A-02-TS2.97f08d10-8897-4a8b-8c35-7eaaaf0a3049', 'TCGA-BH-A0BZ-11A-06-TSF.8926f914-ba66-41ff-b779-574f25cb99b4', 'TCGA-E9-A1NG-11A-01-TSA.ab5cd0d9-cbe1-42a1-9524-6236572cdd37', 'TCGA-E2-A1IU-11A-03-TSC.1e0a5cc1-d90d-4b19-9f96-93f67139fdb7', 'TCGA-E2-A1IU-11A-06-TSF.96609ca3-27d5-47b7-a3b2-9f658b24a26b', 'TCGA-E2-A1IU-11A-02-TSB.b1e1d4f0-7a97-4a67-a15f-667ba0454e31', 'TCGA-E9-A1R7-11A-02-TSB.2e15deaf-7372-447f-a298-f5571d6cc11d']

df = df[~df['slide_name'].isin(names_to_remove)]
df.to_csv(log_tile_extraction_path, index=False)

#### Removing folders given WSI_Names

In [39]:
import shutil
import os
from pathlib import Path

def remove_folders(base_path, folder_names):
    """Remove specified folders from base_path"""
    base_path = Path(base_path)
    
    for folder_name in folder_names:
        folder_path = base_path / folder_name
        if folder_path.exists() and folder_path.is_dir():
            try:
                shutil.rmtree(folder_path)
                print(f"✓ Removed: {folder_path}")
            except Exception as e:
                print(f"✗ Failed to remove {folder_path}: {e}")
        else:
            print(f"⊘ Not found or not a directory: {folder_path}")

# Usage
base_path = "/data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni"
folders_to_remove = ['TCGA-CZ-5467-11A-01-TS1.c634c95c-4d30-46c3-b502-1718b179f560',
'TCGA-B0-5693-11A-01-TS1.e5bd4ec6-e887-41e5-9c58-b4ff5cac6e2c',
'TCGA-BP-4967-11A-01-TS1.72be276e-5622-4a01-a6ef-89780fe64a19',
'TCGA-B0-4847-11A-01-TS1.d340f0ae-ff91-4d8d-83a1-7a730ced005c',
'TCGA-BP-5001-11A-01-TS1.460a8c71-c591-4d95-af1e-2c2fc1124664',
'TCGA-BP-4969-11A-01-TS1.c1c429b7-7769-48b5-b3fd-3dce5aa4c19a']

remove_folders(base_path, folders_to_remove)

⊘ Not found or not a directory: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-CZ-5467-11A-01-TS1.c634c95c-4d30-46c3-b502-1718b179f560
⊘ Not found or not a directory: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-B0-5693-11A-01-TS1.e5bd4ec6-e887-41e5-9c58-b4ff5cac6e2c
✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-BP-4967-11A-01-TS1.72be276e-5622-4a01-a6ef-89780fe64a19
✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-B0-4847-11A-01-TS1.d340f0ae-ff91-4d8d-83a1-7a730ced005c
✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-BP-5001-11A-01-TS1.460a8c71-c591-4d95-af1e-2c2fc1124664
✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_KIRC/Normal_TS/Extracted_Features/uni/TCGA-BP-4969-11A-01-TS1.c1c429b7-7769-48b5-b3fd-3dce5aa4c19a


#### Removing .pt files based on "WSI_Names"

In [64]:
import os
from pathlib import Path

def remove_pt_files(base_path, file_names):
    """Remove .pt files from base_path if they exist"""
    base_path = Path(base_path)
    
    for file_name in file_names:
        # Add .pt extension if not present
        pt_file = file_name if file_name.endswith('.pt') else f"{file_name}.pt"
        file_path = base_path / pt_file
        
        if file_path.exists() and file_path.is_file():
            try:
                file_path.unlink()
                print(f"✓ Removed: {file_path}")
            except Exception as e:
                print(f"✗ Failed to remove {file_path}: {e}")
        else:
            print(f"⊘ Not found or not a file: {file_path}")

# Usage
base_path = "/data_64T_3/Raja/Extracted_Features/TCGA_BRCA/Normal_TS/set1_set2_set3_combined/prov_gigapath"
files_to_remove = ['TCGA-BH-A18Q-11A-03-TSC.fc585bb7-e259-444b-9132-4735a0fbd450',
'TCGA-BH-A1FR-11A-01-TS1.5106fb67-f1cf-4bd6-8946-b76fd3963abe',
'TCGA-BH-A1F8-11B-02-TS2.1b61d7ce-7138-4514-bade-d4a57ff5d900',
'TCGA-E9-A1NG-11A-01-TSA.ab5cd0d9-cbe1-42a1-9524-6236572cdd37']

remove_pt_files(base_path, files_to_remove)

✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_BRCA/Normal_TS/set1_set2_set3_combined/prov_gigapath/TCGA-BH-A18Q-11A-03-TSC.fc585bb7-e259-444b-9132-4735a0fbd450.pt
⊘ Not found or not a file: /data_64T_3/Raja/Extracted_Features/TCGA_BRCA/Normal_TS/set1_set2_set3_combined/prov_gigapath/TCGA-BH-A1FR-11A-01-TS1.5106fb67-f1cf-4bd6-8946-b76fd3963abe.pt
⊘ Not found or not a file: /data_64T_3/Raja/Extracted_Features/TCGA_BRCA/Normal_TS/set1_set2_set3_combined/prov_gigapath/TCGA-BH-A1F8-11B-02-TS2.1b61d7ce-7138-4514-bade-d4a57ff5d900.pt
✓ Removed: /data_64T_3/Raja/Extracted_Features/TCGA_BRCA/Normal_TS/set1_set2_set3_combined/prov_gigapath/TCGA-E9-A1NG-11A-01-TSA.ab5cd0d9-cbe1-42a1-9524-6236572cdd37.pt


##### Copying list of WSI from one location to another location

In [147]:
import shutil
from pathlib import Path

def copy_files(input_folder, output_folder, file_names):
    """Copy specified files from input to output folder, regardless of extension"""
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)
    
    for file_name in file_names:
        # Remove extension if provided
        file_stem = Path(file_name).stem
        
        # Search for files with this stem (any extension)
        matching_files = list(input_path.glob(f"{file_stem}.*"))
        
        if matching_files:
            for src in matching_files:
                dst = output_path / src.name
                try:
                    shutil.copy2(src, dst)
                    print(f"✓ Copied: {src.name}")
                except Exception as e:
                    print(f"✗ Failed to copy {src.name}: {e}")
        else:
            print(f"⊘ Not found: {file_stem}.*")

# Usage
input_folder = "/data_64T_2/Dataset/TCGA_BRCA/images/Tumor_DX1"
output_folder = "/data_64T_3/Raja/Test_WSI"
files_to_copy = ['TCGA-A8-A06T-01Z-00-DX1.BA8B8FC3-6169-48C1-BE1F-37F140CB4D3B',
'TCGA-A8-A09T-01Z-00-DX1.4183BBA4-AF11-4602-9EC2-35083C34A393',
'TCGA-A8-A06O-01Z-00-DX1.FA4495B2-5B13-4448-ADCB-EF5316E0955B',
'TCGA-MS-A51U-01Z-00-DX1.490DE85A-ECE5-4E2A-9657-841BE6FFCCA0',
'TCGA-A8-A09A-01Z-00-DX1.78773A6C-BA8A-46BE-AC2A-9C6FDAB45622',
'TCGA-AO-A0J8-01Z-00-DX1.9BDD4BDE-2A07-4E0C-B146-87365CA9DE3A',
'TCGA-A8-A093-01Z-00-DX1.1C8056D1-11CD-482D-9A23-3A9D1B4E63F0',
'TCGA-HN-A2NL-01Z-00-DX1.C2EAF378-4B37-4C1C-BB0F-18FAC62EEC13',
'TCGA-E2-A574-01Z-00-DX1.60341091-B118-4F20-9ADB-FB2886790B0E',
'TCGA-BH-A1FL-01Z-00-DX1.D3E9F46E-9A28-4AAE-8C87-D6F1851227A3',
'TCGA-BH-A1FB-01Z-00-DX1.9D778D1A-07F6-450D-B7AA-0E17B4D0A88C',
'TCGA-A2-A0CQ-01Z-00-DX1.4E5FB4E5-A08C-4C87-A3BE-0640A95AE649',
'TCGA-D8-A1JL-01Z-00-DX1.FE3F0C6B-F98A-4036-BF9A-25A8CC66B1FD']

copy_files(input_folder, output_folder, files_to_copy)

✓ Copied: TCGA-A8-A06T-01Z-00-DX1.BA8B8FC3-6169-48C1-BE1F-37F140CB4D3B.svs
✓ Copied: TCGA-A8-A09T-01Z-00-DX1.4183BBA4-AF11-4602-9EC2-35083C34A393.svs
✓ Copied: TCGA-A8-A06O-01Z-00-DX1.FA4495B2-5B13-4448-ADCB-EF5316E0955B.svs
✓ Copied: TCGA-MS-A51U-01Z-00-DX1.490DE85A-ECE5-4E2A-9657-841BE6FFCCA0.svs
✓ Copied: TCGA-A8-A09A-01Z-00-DX1.78773A6C-BA8A-46BE-AC2A-9C6FDAB45622.svs
✓ Copied: TCGA-AO-A0J8-01Z-00-DX1.9BDD4BDE-2A07-4E0C-B146-87365CA9DE3A.svs
✓ Copied: TCGA-A8-A093-01Z-00-DX1.1C8056D1-11CD-482D-9A23-3A9D1B4E63F0.svs
✓ Copied: TCGA-HN-A2NL-01Z-00-DX1.C2EAF378-4B37-4C1C-BB0F-18FAC62EEC13.svs
✓ Copied: TCGA-E2-A574-01Z-00-DX1.60341091-B118-4F20-9ADB-FB2886790B0E.svs
✓ Copied: TCGA-BH-A1FL-01Z-00-DX1.D3E9F46E-9A28-4AAE-8C87-D6F1851227A3.svs
✓ Copied: TCGA-BH-A1FB-01Z-00-DX1.9D778D1A-07F6-450D-B7AA-0E17B4D0A88C.svs
✓ Copied: TCGA-A2-A0CQ-01Z-00-DX1.4E5FB4E5-A08C-4C87-A3BE-0640A95AE649.svs
✓ Copied: TCGA-D8-A1JL-01Z-00-DX1.FE3F0C6B-F98A-4036-BF9A-25A8CC66B1FD.svs


In [ ]:
Extract for provgigpath from TCGA_BRCA_Normal_TS
    ['TCGA-BH-A18Q-11A-03-TSC.fc585bb7-e259-444b-9132-4735a0fbd450',
'TCGA-BH-A1FR-11A-01-TS1.5106fb67-f1cf-4bd6-8946-b76fd3963abe',
'TCGA-BH-A1F8-11B-02-TS2.1b61d7ce-7138-4514-bade-d4a57ff5d900',
'TCGA-E9-A1NG-11A-01-TSA.ab5cd0d9-cbe1-42a1-9524-6236572cdd37']
and recreate the labels

In [75]:
import pandas as pd

# Read CSV and find duplicates
df = pd.read_csv('/data_64T_3/Raja/Extracted_Features/TCGA_LUSC/Tumor_DX1/Feature_extraction_metrics_uni.csv')
duplicates = df[df.duplicated('WSI_Name', keep=False)].sort_values('WSI_Name')
print(f"Found {len(duplicates)} duplicate rows:")
print(duplicates)

Found 0 duplicate rows:
Empty DataFrame
Columns: [WSI_Name, Informative_set1, Extracted, Informative_set2, Extracted.1, Informative_set3, Extracted.2, Total_Tiles, Feature_Extr_Time(s), Time_extracted, Additional_note]
Index: []


In [2]:
#!/usr/bin/env python3
"""Ultra-compact folder comparison with hardcoded paths - just configure and run!"""
# ============================================================================
# CONFIGURATION - EDIT THESE PATHS
# ============================================================================
FOLDER1 = "/data_64T_2/Dataset/TCGA_LUAD/masks/CLAM/Normal_TS/tissue_mask"
FOLDER2 = "/data_64T_2/Dataset/TCGA_LUAD/images/Normal_TS"
RECURSIVE = False                     # True for subdirectories, False for top level only
EXTENSIONS = []                       # Filter by extension: ['.svs', '.tif'] or [] for all files
# ============================================================================
# CODE (NO NEED TO EDIT BELOW)
# ============================================================================
from pathlib import Path
f1, f2 = Path(FOLDER1), Path(FOLDER2)
get = lambda d: {(f.relative_to(d).stem if RECURSIVE else f.stem) for f in (d.rglob('*') if RECURSIVE else d.iterdir()) if f.is_file() and (not EXTENSIONS or f.suffix in EXTENSIONS)}
s1, s2 = get(f1), get(f2)
m1, m2 = sorted(s1-s2), sorted(s2-s1)
print(f"Folder 1: {FOLDER1} ({len(s1)} files)")
print(f"Folder 2: {FOLDER2} ({len(s2)} files)")
print(f"Common: {len(s1&s2)}\n")
if m1: print(f"❌ Missing in Folder 2 ({len(m1)}):\n" + "\n".join(f"  {f}" for f in m1) + "\n")
if m2: print(f"❌ Missing in Folder 1 ({len(m2)}):\n" + "\n".join(f"  {f}" for f in m2) + "\n")
print("✅ Identical!" if not (m1 or m2) else f"⚠️  {len(m1)+len(m2)} differences")

Folder 1: /data_64T_2/Dataset/TCGA_LUAD/masks/CLAM/Normal_TS/tissue_mask (183 files)
Folder 2: /data_64T_2/Dataset/TCGA_LUAD/images/Normal_TS (180 files)
Common: 180

❌ Missing in Folder 2 (3):
  TCGA-44-3918-11A-01-TS1.a4fa986d-8039-4994-a1ce-b99403de19b5_1
  TCGA-44-5645-11A-01-TS1.3cc9a6dd-63c1-4c1e-82fb-4ae91e214c0e_1
  TCGA-55-8507-11A-01-TS1.54d196f2-66a7-489f-a227-22864a94bfba_1

⚠️  3 differences


In [156]:
import os
import openslide

def count_wsi_by_mpp(wsi_dir):
    low_mpp = []   # avg_mpp < 0.30
    high_mpp = []  # avg_mpp > 0.40

    for fname in os.listdir(wsi_dir):
        if not fname.lower().endswith('.svs'):
            continue

        fpath = os.path.join(wsi_dir, fname)

        try:
            slide = openslide.OpenSlide(fpath)
            prop = slide.properties

            mpp_x = float(prop.get('openslide.mpp-x'))
            mpp_y = float(prop.get('openslide.mpp-y'))
            avg_mpp = (mpp_x + mpp_y) / 2

            if 0.20 > avg_mpp < 0.30:
                low_mpp.append(fname)
            elif avg_mpp > 0.40:
                high_mpp.append(fname)

            slide.close()

        except Exception as e:
            print(f"[WARN] Skipping {fname}: {e}")

    print(f"Total WSIs checked      : {len(low_mpp) + len(high_mpp)}")
    print(f"avg_mpp < 0.30 (≈40×)   : {len(low_mpp)}")
    print(f"avg_mpp > 0.40 (≈20×)   : {len(high_mpp)}")

    return low_mpp, high_mpp

count_wsi_by_mpp("/data_64T_2/Dataset/TCGA_STAD/images/Tumor_DX1")

[WARN] Skipping TCGA-CG-5726-01Z-00-DX1.AF4BAEE4-45CE-453F-B4B9-8373A5909315.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5723-01Z-00-DX1.A0FB6EFA-304F-4267-8C6E-D9E750F67FDC.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5717-01Z-00-DX1.6E375046-B9C3-49DD-A934-9AE4BE2895E0.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5724-01Z-00-DX1.89B75EBF-BF6E-418F-8FA7-F564055EE6DA.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5722-01Z-00-DX1.C1B50B01-0328-4DAF-8EDA-024D2A7EC07F.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5725-01Z-00-DX1.2FDB162B-9080-434D-AEE0-B867BE8D3786.svs: float() argument must be a string or a number, not 'NoneType'
[WARN] Skipping TCGA-CG-5730-01Z-00-DX1.FA6F1EEE-3581-4824-AFC2-A5FFC04DF5A2.svs: float() argument must be a string or a number, not 'No

(['TCGA-R5-A7ZI-01Z-00-DX1.3A4A56F1-1CFD-441D-85F3-52DDA0B346A5.svs',
  'TCGA-D7-A74A-01Z-00-DX1.ED9A8AA2-7D69-462C-8544-587783F15DB5.svs',
  'TCGA-BR-8679-01Z-00-DX1.2aa8a10d-4b62-479b-bea8-6c48428a888a.svs',
  'TCGA-HJ-7597-01Z-00-DX1.4218e9a0-3ef1-4073-8336-1fe81c0bce90.svs',
  'TCGA-VQ-A94P-01Z-00-DX1.83E87D6C-D210-4C06-BB5B-49A1F3E14EE0.svs',
  'TCGA-CD-8529-01Z-00-DX1.C3BA5482-04EB-4E1A-BFFA-5461489B5792.svs',
  'TCGA-VQ-A8PM-01Z-00-DX1.49FE7D00-2547-4936-803D-F4421C5E5BB3.svs',
  'TCGA-CD-A489-01Z-00-DX1.AB03DA27-6A86-41C7-9305-E5400E6C47AC.svs',
  'TCGA-RD-A8N2-01Z-00-DX1.A56448E6-1C5E-4758-AC2E-4591A406116C.svs',
  'TCGA-BR-8485-01Z-00-DX1.863bea72-3bd3-4f07-bbc9-e00d17ae39f7.svs',
  'TCGA-D7-A4YT-01Z-00-DX1.645AF39D-1B70-4C46-AB80-7B532994EE66.svs',
  'TCGA-BR-8289-01Z-00-DX1.3fd11755-cb87-40fe-92e2-0ccc55aefa0d.svs',
  'TCGA-CG-4444-01Z-00-DX1.2f966933-a92c-4888-853c-2f8ca195120e.svs',
  'TCGA-BR-8680-01Z-00-DX1.a314ef6e-f3f9-4148-934e-dc432e3f39e8.svs',
  'TCGA-D7-A6EZ-01Z-

In [34]:
import openslide
slide_path = r'/data_64T_2/Dataset/TCGA_BLCA/images/Tumor_DX1/TCGA-XF-AAMF-01Z-00-DX1.01371D44-969E-44C8-993B-13B83D9E4F0F.svs'   
slide = openslide.OpenSlide(slide_path)
# Retrieve and display properties line by line
print("WSI Properties:\n")
for key, value in slide.properties.items():
    print(f"{key}: {value}")

WSI Properties:

aperio.AppMag: 40
aperio.DSR ID: resc3-dsr1
aperio.Date: 08/06/14
aperio.DisplayColor: 0
aperio.Exposure Scale: 0.000001
aperio.Exposure Time: 109
aperio.Filename: TCGA-XF-AAMF-01Z-00-DX1
aperio.Focus Offset: 0.000000
aperio.ICC Profile: ScanScope v1
aperio.ImageID: 160244
aperio.Left: 18.834389
aperio.LineAreaXOffset: 0.011464
aperio.LineAreaYOffset: -0.002805
aperio.LineCameraSkew: -0.000153
aperio.MPP: 0.2527
aperio.OriginalHeight: 58186
aperio.OriginalWidth: 136144
aperio.Parmset: GOG136 on O: Drive
aperio.ScanScope ID: SS1764CNTLR
aperio.StripeWidth: 2032
aperio.Time: 20:09:16
aperio.Time Zone: GMT-04:00
aperio.Title: TCGA-XF-AAMF-01Z-00-DX1
aperio.Top: 20.351276
aperio.User: bd76b653-c791-4904-875f-b678a009dde6
openslide.comment: Aperio Image Library v12.0.15 
136144x58186 [0,100 133463x58086] (240x240) JPEG/RGB Q=30|AppMag = 40|StripeWidth = 2032|ScanScope ID = SS1764CNTLR|Filename = TCGA-XF-AAMF-01Z-00-DX1|Title = TCGA-XF-AAMF-01Z-00-DX1|Date = 08/06/14|Time = 

In [67]:
import openslide
slide_path = r'/data_64T_2/Dataset/TCGA_BRCA/images/Tumor_DX1/TCGA-OL-A66H-01Z-00-DX1.E54AF3FA-E59E-404C-BB83-A6FC6FC9B312.svs'   
slide = openslide.OpenSlide(slide_path)
# Retrieve and display properties line by line
print("WSI Properties:\n")
for key, value in slide.properties.items():
    print(f"{key}: {value}")

WSI Properties:

aperio.AppMag: 40
aperio.MPP: 0.11625
openslide.comment: Aperio Image Library v11.0.37
167936x420096 (256x256) JPEG/RGB Q=40;Mirax Digital Slide|AppMag = 40|MPP = 0.11625
openslide.level-count: 5
openslide.level[0].downsample: 1
openslide.level[0].height: 420096
openslide.level[0].tile-height: 256
openslide.level[0].tile-width: 256
openslide.level[0].width: 167936
openslide.level[1].downsample: 4
openslide.level[1].height: 105024
openslide.level[1].tile-height: 256
openslide.level[1].tile-width: 256
openslide.level[1].width: 41984
openslide.level[2].downsample: 16
openslide.level[2].height: 26256
openslide.level[2].tile-height: 256
openslide.level[2].tile-width: 256
openslide.level[2].width: 10496
openslide.level[3].downsample: 64
openslide.level[3].height: 6564
openslide.level[3].tile-height: 256
openslide.level[3].tile-width: 256
openslide.level[3].width: 2624
openslide.level[4].downsample: 256
openslide.level[4].height: 1641
openslide.level[4].tile-height: 256
opens

In [30]:
import openslide
slide_path = r'/data_64T_2/Dataset/STANFORD_793/images/Tumor/P-0001.svs'   
slide = openslide.OpenSlide(slide_path)
# Retrieve and display properties line by line
print("WSI Properties:\n")
for key, value in slide.properties.items():
    print(f"{key}: {value}")

WSI Properties:

aperio.AppMag: 40
aperio.Date: 06/19/18
aperio.DisplayColor: 0
aperio.Exposure Scale: 0.000001
aperio.Exposure Time: 45
aperio.Filename: 1000697
aperio.Focus Offset: -0.000500
aperio.ICC Profile: AT2
aperio.ImageID: 1000697
aperio.Left: 21.563229
aperio.LineAreaXOffset: 0.005249
aperio.LineAreaYOffset: -0.006157
aperio.LineCameraSkew: -0.001190
aperio.MPP: 0.2524
aperio.OriginalHeight: 87981
aperio.OriginalWidth: 101600
aperio.ScanScope ID: LEICA-SCAN-W01
aperio.SessonMode: NR
aperio.StripeWidth: 2032
aperio.Time: 02:49:28
aperio.Time Zone: GMT-07:00
aperio.Top: 22.992769
aperio.User: 548b3db4-8219-4332-ae4a-eee65396a0dd
openslide.comment: Aperio Image Library v12.0.15 
101600x87981 [0,100 99600x87881] (240x240) JPEG/RGB Q=70|AppMag = 40|StripeWidth = 2032|ScanScope ID = LEICA-SCAN-W01|Filename = 1000697|Date = 06/19/18|Time = 02:49:28|Time Zone = GMT-07:00|User = 548b3db4-8219-4332-ae4a-eee65396a0dd|MPP = 0.2524|Left = 21.563229|Top = 22.992769|LineCameraSkew = -0.001

In [38]:
slide.properties.get('openslide.comment') 

'Aperio Image Library v11.0.37\r\n81152x102656 (256x256) JPEG/RGB Q=40;Mirax Digital Slide|AppMag = 40|MPP = 0.16437'

In [45]:
slide.properties.get('aperio.MPP')

'0.16437'

In [20]:
import os
import openslide
from pathlib import Path

folder = "/data_64T_2/Dataset/STANFORD_793/images"
wsi_files = list(Path(folder).glob("*.svs"))

inch_count = sum(1 for f in wsi_files 
                 if openslide.OpenSlide(str(f)).properties.get('tiff.ResolutionUnit', '').lower() == 'inch')

print(f"Total SVS files: {len(wsi_files)}")
print(f"Files with ResolutionUnit='inch': {inch_count}")

Total SVS files: 793
Files with ResolutionUnit='inch': 793


In [ ]:
import os
import openslide
from pathlib import Path

folder = "/data_64T_2/Dataset/STANFORD_793/images"
wsi_files = list(Path(folder).glob("*.svs"))

inch_count = 0
for wsi_path in wsi_files:
    slide = openslide.OpenSlide(str(wsi_path))
    for key, value in slide.properties.items():
        if key == 'tiff.ResolutionUnit' and 'inch' in value.lower():
            inch_count += 1
            break
    slide.close()

print(f"Total SVS files: {len(wsi_files)}")
print(f"Files with ResolutionUnit='inch': {inch_count}")

In [61]:
import re
text = "tiff.ImageDescription: Aperio Image Library v11.0.37 81152x102656 (256x256) JPEG/RGB Q=40;Mirax Digital Slide|AppMag =40|MPP=0.16437"
val = re.search(r"\|?MPP\s*=\s*([\d\.]+)", text)

In [62]:
val.group(1)

'0.16437'

In [64]:
if val and val.group(1):
    print("yes")

yes


### Getting full name of a slide from its patiend id

In [3]:
import pandas as pd
import os

def map_slide_files(csv_path, search_dir, output_path):
    df = pd.read_csv(csv_path)
    # Get all filenames from the directory once for efficiency
    all_files = os.listdir(search_dir)
    
    # Define a simple lambda to find the first file containing the slide_name substring
    df['full_file_name'] = df['slide_name'].apply(
        lambda x: next((f for f in all_files if str(x) in f), None)
    )
    
    df.to_csv(output_path, index=False)
    print(f"Mapping complete. File saved to: {output_path}")

# --- Set your paths here ---
csv_input = '/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/MSS_MSI/COADREAD_MSI_MSS_labels.csv' 
slides_folder = '/data_64T_2/Dataset/TCGA_READ/images/Tumor_DX1'
csv_output = '/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/MSS_MSI/COADREAD_MSI_MSS_labels_new1.csv'

map_slide_files(csv_input, slides_folder, csv_output)

Mapping complete. File saved to: /home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/MSS_MSI/COADREAD_MSI_MSS_labels_new1.csv


In [4]:
import pandas as pd

# Load the CSV
df = pd.read_csv('/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/MSS_MSI/COADREAD_MSI_MSS_labels_new.csv')

# Remove the extension by splitting from the right at the last dot
df['slide_name'] = df['slide_name'].str.rsplit('.', n=1).str[0]

# Save the updated CSV
df.to_csv('/home/rajaj/Project/7.WSI_Analysis_Experiments/1.WSI_Classification/datasets/MSS_MSI/COADREAD_MSI_MSS_labels_new1.csv', index=False)